# MatchFormer Epipolar Fine-tuning — Google Colab

This notebook fine-tunes MatchFormer with epipolar supervision baked in.  
Checkpoints are saved to **Google Drive** so training can be resumed if Colab disconnects.

**Before running:**
1. Set Runtime → Change runtime type → **T4 GPU**
2. Upload your ScanNet data to Google Drive (see Cell 3)
3. Run cells **top to bottom** — on resume, skip to Cell 6

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints will be saved here — survives Colab restarts
DRIVE_CKPT_DIR = '/content/drive/MyDrive/final_proj/matchformer_checkpoints_run2'

import os
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {DRIVE_CKPT_DIR}')

## Cell 2: Clone Repo & Install Dependencies

In [11]:
import os

REPO_URL = 'https://github.com/sid-raj-uc/match-former.git'  # your repo
REPO_DIR = '/content/final_proj'

if not os.path.exists(REPO_DIR):
    print('Repo not cloned — cloning')
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already cloned — pulling latest')
    !cd {REPO_DIR} && git pull

%cd /content/final_proj/MatchFormer

Repo already cloned — pulling latest
remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 759 bytes | 94.00 KiB/s, done.
From https://github.com/sid-raj-uc/match-former
   72dbf44..3887e1f  main       -> origin/main
Updating 72dbf44..3887e1f
Fast-forward
 MatchFormer/model/datasets/scannet_simple.py | 2 ++
 MatchFormer/train_finetune.py                | 4 +++-
 2 files changed, 5 insertions(+), 1 deletion(-)
/content/final_proj/MatchFormer


In [3]:
# Install dependencies
!pip install -q pytorch-lightning einops kornia timm loguru yacs
!pip install -q opencv-python-headless
print('Dependencies installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 15.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 108.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.8 MB/s eta 0:00:00
Dependencies installed.


## Cell 3: ScanNet Data

Upload your exported ScanNet scenes to Google Drive. Expected structure:
```
MyDrive/final_proj/scannet/scans/
  scene0000_00/exported/{color, depth, pose, intrinsic}
  scene0001_00/exported/{color, depth, pose, intrinsic}
  ...
```
The training script picks up **all scene subdirs automatically** — no config changes needed.

In [ ]:
# Point to the root scans directory — all scene subdirs will be used automatically
# Expected layout on Drive:
#   MyDrive/final_proj/scannet/scans/
#     scene0000_00/exported/{color, depth, pose, intrinsic}
#     scene0001_00/exported/{color, depth, pose, intrinsic}
#     ...
DATA_ROOT = '/content/drive/MyDrive/final_proj/scannet/scans'

import os, glob
scenes = sorted([d for d in os.listdir(DATA_ROOT)
                 if os.path.isdir(os.path.join(DATA_ROOT, d, 'exported', 'color'))])
print(f'Found {len(scenes)} scenes:')
total_frames = 0
for s in scenes:
    n = len(glob.glob(os.path.join(DATA_ROOT, s, 'exported', 'color', '*.jpg')))
    total_frames += n
    print(f'  {s}: {n} frames')
print(f'\nTotal frames: {total_frames}')

## Cell 4: Download Pretrained Weights

In [5]:
# Check if pretrained weight already exists (from a previous run or Drive)
WEIGHTS_PATH = 'model/weights/indoor-lite-LA.ckpt'
DRIVE_WEIGHTS = '/content/drive/MyDrive/final_proj/MatchFormer/model/weights/indoor-lite-LA.ckpt'

os.makedirs('model/weights', exist_ok=True)

if os.path.exists(DRIVE_WEIGHTS) and not os.path.exists(WEIGHTS_PATH):
    import shutil
    shutil.copy(DRIVE_WEIGHTS, WEIGHTS_PATH)
    print('Copied weights from Drive')
elif os.path.exists(WEIGHTS_PATH):
    print('Weights already present')
else:
    print('⚠️  Please upload indoor-lite-LA.ckpt to Drive at:', DRIVE_WEIGHTS)
    print('   or place it at:', WEIGHTS_PATH)

Copied weights from Drive


## Cell 5: Verify GPU & Run Sanity Check
Run this once to confirm the pipeline works before doing the full training.

In [6]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (no GPU!)"}')
print(f'CUDA available: {torch.cuda.is_available()}')

GPU: Tesla T4
CUDA available: True


In [ ]:
# Sanity check: 5 pairs, 50 steps — should finish in ~1 min on L4
!python train_finetune.py \
    --overfit \
    --data_dir {DATA_ROOT} \
    --checkpoint_dir {DRIVE_CKPT_DIR}/overfit \
    --batch 4 \
    --steps 50

print('Sanity check complete — loss should be decreasing!')

## Cell 6: Full Fine-tuning

**Auto-resume is enabled** — if Colab disconnects, just re-run from Cell 1 and then run this cell again. It will automatically pick up from the last saved checkpoint in your Drive.

**Key hyperparameters:**
- `--steps 10000` — total training steps (~2-3 hours on T4)
- `--tau 10.0` — epipolar constraint strength
- `--save_every 200` — save checkpoint to Drive every 200 steps
- `--batch 4` — batch size (reduce to 2 if OOM)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# L4 GPU: 24GB VRAM — batch 4, tau 50 (soft epipolar mask for robust multi-scene training)
# tau=50 is broader than tau=10 — prevents confidence collapse across diverse scenes
!python train_finetune.py \
    --data_dir {DATA_ROOT} \
    --checkpoint_dir {DRIVE_CKPT_DIR} \
    --steps 10000 \
    --tau 50.0 \
    --batch 4 \
    --workers 4 \
    --lr 1e-4 \
    --save_every 200

# To resume from last checkpoint (Colab reconnect):
# !python train_finetune.py \
#     --data_dir {DATA_ROOT} \
#     --checkpoint_dir {DRIVE_CKPT_DIR} \
#     --steps 10000 \
#     --tau 50.0 --batch 4 --workers 4 --save_every 200

## Cell 7: Run Benchmark After Training

In [13]:
# After training finishes, copy the best checkpoint locally and run benchmark
import shutil

TRAINED_CKPT = f'{DRIVE_CKPT_DIR}/last.ckpt'
shutil.copy(TRAINED_CKPT, 'model/weights/epipolar-finetuned.ckpt')
print('Checkpoint copied.')

# Run benchmark with finetuned model
!python run_benchmark.py \
    --num_pairs 100 \
    --ckpt model/weights/epipolar-finetuned.ckpt \
    2>&1 | tee benchmark_finetuned_log.txt

# Copy results back to Drive
shutil.copy('benchmark_finetuned_log.txt', f'{DRIVE_CKPT_DIR}/benchmark_finetuned_log.txt')
print('Benchmark results saved to Drive!')

Checkpoint copied.
/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
usage: run_benchmark.py [-h] [--num_pairs NUM_PAIRS]
run_benchmark.py: error: unrecognized arguments: --ckpt model/weights/epipolar-finetuned.ckpt
Benchmark results saved to Drive!
